# 14b — Read-only audit: historical vs current CopyKAT

This notebook is strictly read-only. It does not modify any project file and does not write any new H5AD or TSV output.

Objectives:

- Perform a cell-by-cell comparison between the historical and current CopyKAT results.
- Quantify exact call transitions such as `diploid -> aneuploid`.
- Compare differences across clusters, marker-based annotations, datasets, and conditions.
- Determine whether the increase in aneuploid calls is concentrated in tumor-like cells or also affects reference/non-malignant-like compartments.
- Provide a decision-oriented audit before proceeding to Notebook 15.

Important: this notebook does not automatically choose a preferred CopyKAT run. It organizes the evidence so that the final choice can be made on scientific grounds.

**DS2 identity handling:** raw historical and current cell IDs differ for all 21,391 DS2 cells. Stable DS2 identity is therefore defined as normalized `sample_id` plus the nucleotide barcode extracted from `cell_id`. DS1, DS3, and DS4 continue to use normalized direct cell IDs.

**Two missing-cell identity handling:** the two rows with missing CopyKAT calls also lack dataset/sample metadata. They are matched only by normalized raw H5AD `cell_id`, without assigning biological metadata. The audit remains strictly read-only.


In [ ]:
from pathlib import Path
import os
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)

def find_repo(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for p in (start, *start.parents):
        if (p / "README.md").exists() and (p / "metadata").exists():
            return p
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", find_repo())
).expanduser().resolve()

METADATA_DIR = PROJECT_DIR / "metadata"
HIST_DIR = METADATA_DIR / "historical_copykat"

CURRENT_FINAL = METADATA_DIR / "copykat_microbatch_final_cell_calls_HVG8000_r0_6.tsv"
CURRENT_CLUSTER = METADATA_DIR / "copykat_microbatch_cluster_summary_HVG8000_r0_6.tsv"
CURRENT_OVERALL = METADATA_DIR / "copykat_microbatch_overall_summary_HVG8000_r0_6.tsv"

HIST_FINAL = HIST_DIR / "copykat_microbatch_final_cell_calls_HVG8000_r0_6.tsv"
HIST_CLUSTER = HIST_DIR / "copykat_microbatch_cluster_summary_HVG8000_r0_6.tsv"
HIST_OVERALL = HIST_DIR / "copykat_microbatch_overall_summary_HVG8000_r0_6.tsv"

required = [
    CURRENT_FINAL,
    CURRENT_CLUSTER,
    CURRENT_OVERALL,
    HIST_FINAL,
    HIST_CLUSTER,
    HIST_OVERALL,
]

print("PROJECT_DIR:", PROJECT_DIR)
print("HIST_DIR:", HIST_DIR)

missing = [p for p in required if not p.exists()]

if missing:
    raise FileNotFoundError(
        "Missing required file(s):\n" + "\n".join(map(str, missing))
    )

print("\nAll six required comparison files found.")


In [ ]:
# =========================
# LOAD FILES
# =========================
cur = pd.read_csv(CURRENT_FINAL, sep="\t")
hist = pd.read_csv(HIST_FINAL, sep="\t")

cur_cluster = pd.read_csv(CURRENT_CLUSTER, sep="\t")
hist_cluster = pd.read_csv(HIST_CLUSTER, sep="\t")

cur_overall = pd.read_csv(CURRENT_OVERALL, sep="\t")
hist_overall = pd.read_csv(HIST_OVERALL, sep="\t")

print("Current final:", cur.shape)
print("Historical final:", hist.shape)
print("Current cluster:", cur_cluster.shape)
print("Historical cluster:", hist_cluster.shape)

display(
    pd.DataFrame({
        "file": ["current", "historical"],
        "n_rows": [len(cur), len(hist)],
        "n_unique_cell_id": [
            cur["cell_id"].astype(str).nunique(),
            hist["cell_id"].astype(str).nunique(),
        ],
    })
)


In [ ]:
# =========================
# DATASET-AWARE STABLE CELL IDENTIFIERS
# =========================
#
# Historical/current raw cell IDs differ systematically for DS2_GSE141383.
# This is NOT a biological mismatch:
# 21,391 unmatched IDs == the complete DS2 cell count.
#
# We therefore use the same stable biological-cell identity logic that was
# previously validated during the historical-vs-current scVI audit:
#
#   DS1 / DS3 / DS4:
#       normalized cell_id is exact.
#
#   DS2:
#       normalized sample_id + nucleotide barcode extracted from cell_id.
#
# Historical dataset aliases DS5 -> DS3 and DS13 -> DS4 are normalized too.
# This cell is READ-ONLY.

def normalize_dataset(value):
    value = str(value)

    return (
        value
        .replace(
            "DS5_GSE182109",
            "DS3_GSE182109",
        )
        .replace(
            "DS13_GSE278450",
            "DS4_GSE278450",
        )
    )


def normalize_obs_name(value):
    value = str(value)

    return (
        value
        .replace(
            "DS5_GSE182109",
            "DS3_GSE182109",
        )
        .replace(
            "DS13_GSE278450",
            "DS4_GSE278450",
        )
    )


def normalize_sample(value):
    # Historical DS2 sample labels sometimes carry a leading GSM accession,
    # whereas the current pipeline may store the biological sample label alone.
    return re.sub(
        r"^GSM\d+_",
        "",
        str(value),
    )


def extract_nucleotide_barcode(value):
    candidates = re.findall(
        r"[ACGTN]{8,30}",
        str(value).upper(),
    )

    if not candidates:
        return None

    return max(
        candidates,
        key=len,
    )


def normalize_call(value):
    if pd.isna(value):
        return "missing"

    value = str(value).strip().lower()

    mapping = {
        "aneuploid": "aneuploid",
        "diploid": "diploid",
        "not.defined": "not.defined",
        "not_defined": "not.defined",
        "not defined": "not.defined",
        "missing": "missing",
        "nan": "missing",
    }

    return mapping.get(
        value,
        value,
    )


def canonical_cell_keys(df, label):
    required = [
        "cell_id",
        "core_dataset",
        "sample_id",
    ]

    missing = [
        col
        for col in required
        if col not in df.columns
    ]

    if missing:
        raise RuntimeError(
            f"{label}: missing required column(s) for stable identity: "
            + ", ".join(missing)
        )

    datasets = (
        df["core_dataset"]
        .astype(str)
        .map(normalize_dataset)
    )

    keys = pd.Series(
        index=df.index,
        dtype="object",
    )

    # --------------------------------------------------------
    # DS1 / DS3 / DS4:
    # direct normalized cell IDs were previously shown to be exact.
    # --------------------------------------------------------
    for dataset in [
        "DS1_GSE103224",
        "DS3_GSE182109",
        "DS4_GSE278450",
    ]:
        mask = datasets.eq(
            dataset
        )

        keys.loc[mask] = [
            dataset
            + "||"
            + normalize_obs_name(
                cell_id
            )

            for cell_id
            in df.loc[
                mask,
                "cell_id",
            ].astype(str)
        ]

    # --------------------------------------------------------
    # DS2:
    # sample label + nucleotide barcode is the stable identity.
    # --------------------------------------------------------
    ds2_mask = datasets.eq(
        "DS2_GSE141383"
    )

    ds2_samples = (
        df.loc[
            ds2_mask,
            "sample_id",
        ]
        .astype(str)
        .map(normalize_sample)
    )

    ds2_barcodes = (
        df.loc[
            ds2_mask,
            "cell_id",
        ]
        .astype(str)
        .map(
            extract_nucleotide_barcode
        )
    )

    if ds2_barcodes.isna().any():
        examples = (
            df.loc[
                ds2_mask
                & ds2_barcodes.isna(),
                "cell_id",
            ]
            .astype(str)
            .head(10)
            .tolist()
        )

        raise RuntimeError(
            f"{label}: DS2 contains "
            f"{int(ds2_barcodes.isna().sum())} cells without an "
            "extractable nucleotide barcode. Examples: "
            + repr(examples)
        )

    keys.loc[
        ds2_mask
    ] = (
        "DS2_GSE141383||"
        + ds2_samples
        + "||"
        + ds2_barcodes
    )

    # --------------------------------------------------------
    # Two cells have no target CopyKAT prediction and therefore also
    # lack dataset/sample metadata in the integrated final-call table.
    # Historical notebook 14 also contains exactly two missing calls.
    #
    # For these rows only, use normalized raw H5AD obs_name as the
    # stable identity. This does NOT assign any biological metadata.
    # --------------------------------------------------------
    unresolved = keys.isna()

    if unresolved.any():
        print(
            f"{label}: unresolved metadata rows =",
            int(unresolved.sum()),
        )

        preview_cols = [
            col
            for col in [
                "cell_id",
                "core_dataset",
                "sample_id",
                "copykat_microbatch_pred",
            ]
            if col in df.columns
        ]

        display(
            df.loc[
                unresolved,
                preview_cols,
            ]
        )

        fallback_ids = (
            df.loc[
                unresolved,
                "cell_id",
            ]
            .astype(str)
            .map(normalize_obs_name)
        )

        keys.loc[
            unresolved
        ] = (
            "UNASSIGNED_METADATA||"
            + fallback_ids
        )

    if keys.duplicated().any():
        duplicate_examples = (
            keys[
                keys.duplicated(
                    keep=False
                )
            ]
            .head(20)
            .tolist()
        )

        raise RuntimeError(
            f"{label}: canonical biological-cell keys are not unique. "
            f"Examples: {duplicate_examples}"
        )

    return keys


# Keep raw IDs for diagnostics.
cur["cell_id_raw"] = (
    cur["cell_id"]
    .astype(str)
)

hist["cell_id_raw"] = (
    hist["cell_id"]
    .astype(str)
)

cur[
    "copykat_call_norm"
] = (
    cur[
        "copykat_microbatch_pred"
    ]
    .map(
        normalize_call
    )
)

hist[
    "copykat_call_norm"
] = (
    hist[
        "copykat_microbatch_pred"
    ]
    .map(
        normalize_call
    )
)

# First show the raw-ID mismatch by dataset.
cur_raw_ids = set(
    cur[
        "cell_id_raw"
    ]
)

hist_raw_ids = set(
    hist[
        "cell_id_raw"
    ]
)

print("=" * 80)
print("RAW CELL-ID AUDIT")
print("=" * 80)

print(
    "Raw shared IDs:",
    len(
        cur_raw_ids
        & hist_raw_ids
    ),
)

print(
    "Raw current-only IDs:",
    len(
        cur_raw_ids
        - hist_raw_ids
    ),
)

print(
    "Raw historical-only IDs:",
    len(
        hist_raw_ids
        - cur_raw_ids
    ),
)

print(
    "\nExpected DS2 total cells: 21,391"
)

# Dataset-aware stable keys.
cur[
    "cell_id_canonical"
] = canonical_cell_keys(
    cur,
    "current",
)

hist[
    "cell_id_canonical"
] = canonical_cell_keys(
    hist,
    "historical",
)

cur_ids = set(
    cur[
        "cell_id_canonical"
    ]
)

hist_ids = set(
    hist[
        "cell_id_canonical"
    ]
)

print("\n" + "=" * 80)
print("STABLE BIOLOGICAL CELL-ID AUDIT")
print("=" * 80)

print(
    "Current canonical unique:",
    len(
        cur_ids
    ),
)

print(
    "Historical canonical unique:",
    len(
        hist_ids
    ),
)

print(
    "Shared canonical IDs:",
    len(
        cur_ids
        & hist_ids
    ),
)

print(
    "Current-only canonical IDs:",
    len(
        cur_ids
        - hist_ids
    ),
)

print(
    "Historical-only canonical IDs:",
    len(
        hist_ids
        - cur_ids
    ),
)

if (
    len(
        cur_ids
    )
    != 351427
):
    raise RuntimeError(
        "Current canonical cell IDs are not exactly 351,427 unique cells."
    )

if (
    len(
        hist_ids
    )
    != 351427
):
    raise RuntimeError(
        "Historical canonical cell IDs are not exactly 351,427 unique cells."
    )

if cur_ids != hist_ids:
    print(
        "\nCurrent-only examples:"
    )
    print(
        list(
            sorted(
                cur_ids
                - hist_ids
            )
        )[:20]
    )

    print(
        "\nHistorical-only examples:"
    )
    print(
        list(
            sorted(
                hist_ids
                - cur_ids
            )
        )[:20]
    )

    raise RuntimeError(
        "Stable biological cell sets still differ after DS2-aware canonicalization."
    )

print(
    "\nPASS: historical and current CopyKAT final-call tables "
    "represent the exact same 351,427 biological cells."
)


In [ ]:
# =========================
# CELL-BY-CELL MERGE
# =========================
meta_candidates = [
    "leiden_scvi_main",
    "annotation_marker_based_HVG8000_r0_6",
    "malignancy_status_marker_based_HVG8000_r0_6",
    "lineage_marker_based_HVG8000_r0_6",
    "cnv_role_suggestion_HVG8000_r0_6",
    "core_dataset",
    "condition",
    "patient_id",
    "sample_id",
]

cur_meta_cols = [
    c for c in meta_candidates
    if c in cur.columns
]

cur_small = cur[
    [
        "cell_id_canonical",
        "copykat_call_norm",
        *cur_meta_cols,
    ]
].copy().rename(
    columns={"copykat_call_norm": "current_call"}
)

hist_small = hist[
    [
        "cell_id_canonical",
        "copykat_call_norm",
    ]
].copy().rename(
    columns={"copykat_call_norm": "historical_call"}
)

cmp = hist_small.merge(
    cur_small,
    on="cell_id_canonical",
    how="outer",
    validate="one_to_one",
    indicator=True,
)

cmp["historical_call"] = cmp["historical_call"].fillna("missing")
cmp["current_call"] = cmp["current_call"].fillna("missing")
cmp["changed"] = cmp["historical_call"] != cmp["current_call"]

print("Merged comparison:", cmp.shape)
print("Changed cells:", int(cmp["changed"].sum()))
print("Unchanged cells:", int((~cmp["changed"]).sum()))

display(
    cmp["_merge"]
    .value_counts(dropna=False)
    .rename_axis("merge_status")
    .reset_index(name="n")
)


In [ ]:
# =========================
# OVERALL TRANSITION MATRIX
# =========================
call_order = [
    "aneuploid",
    "diploid",
    "not.defined",
    "missing",
]

transition = pd.crosstab(
    cmp["historical_call"],
    cmp["current_call"],
    dropna=False,
).reindex(
    index=call_order,
    columns=call_order,
    fill_value=0,
)

print("Cell-by-cell transition matrix")
display(transition)

transition_long = (
    cmp.groupby(
        ["historical_call", "current_call"],
        observed=True,
    )
    .size()
    .rename("n_cells")
    .reset_index()
    .sort_values("n_cells", ascending=False)
)

display(transition_long)

key_transitions = pd.DataFrame([
    {
        "transition": "diploid -> aneuploid",
        "n_cells": int(
            (
                (cmp["historical_call"] == "diploid")
                & (cmp["current_call"] == "aneuploid")
            ).sum()
        ),
    },
    {
        "transition": "aneuploid -> diploid",
        "n_cells": int(
            (
                (cmp["historical_call"] == "aneuploid")
                & (cmp["current_call"] == "diploid")
            ).sum()
        ),
    },
    {
        "transition": "not.defined -> aneuploid",
        "n_cells": int(
            (
                (cmp["historical_call"] == "not.defined")
                & (cmp["current_call"] == "aneuploid")
            ).sum()
        ),
    },
    {
        "transition": "not.defined -> diploid",
        "n_cells": int(
            (
                (cmp["historical_call"] == "not.defined")
                & (cmp["current_call"] == "diploid")
            ).sum()
        ),
    },
    {
        "transition": "any changed",
        "n_cells": int(cmp["changed"].sum()),
    },
])

display(key_transitions)


In [ ]:
# =========================
# OVERALL CALL FRACTIONS
# =========================
def call_summary(df, label):
    counts = (
        df.value_counts(dropna=False)
        .reindex(call_order, fill_value=0)
    )

    n_defined = int(
        counts.get("aneuploid", 0)
        + counts.get("diploid", 0)
    )

    return {
        "run": label,
        "aneuploid": int(counts.get("aneuploid", 0)),
        "diploid": int(counts.get("diploid", 0)),
        "not.defined": int(counts.get("not.defined", 0)),
        "missing": int(counts.get("missing", 0)),
        "defined": n_defined,
        "aneuploid_fraction_defined": (
            counts.get("aneuploid", 0) / n_defined
            if n_defined else np.nan
        ),
    }

overall_compare = pd.DataFrame([
    call_summary(cmp["historical_call"], "historical"),
    call_summary(cmp["current_call"], "current"),
])

display(overall_compare)

hist_frac = overall_compare.loc[
    overall_compare["run"] == "historical",
    "aneuploid_fraction_defined",
].iloc[0]

cur_frac = overall_compare.loc[
    overall_compare["run"] == "current",
    "aneuploid_fraction_defined",
].iloc[0]

print(
    "Defined-cell aneuploid fraction change:",
    f"{hist_frac:.4f} -> {cur_frac:.4f}",
    "| delta percentage points =",
    round((cur_frac - hist_frac) * 100, 3),
)


In [ ]:
# =========================
# BROAD BIOLOGICAL GROUPS
# =========================
ANNOT_COL = (
    "annotation_marker_based_HVG8000_r0_6"
    if "annotation_marker_based_HVG8000_r0_6" in cmp.columns
    else None
)

CLUSTER_COL = (
    "leiden_scvi_main"
    if "leiden_scvi_main" in cmp.columns
    else None
)

DATASET_COL = (
    "core_dataset"
    if "core_dataset" in cmp.columns
    else None
)

CONDITION_COL = (
    "condition"
    if "condition" in cmp.columns
    else None
)


def broad_biology(annotation):
    s = str(annotation).lower()

    if s in {"nan", "none", ""}:
        return "missing_annotation"

    if (
        "putative_tumor" in s
        or "cycling_proliferating" in s
    ):
        return "tumor_like_or_cycling"

    if any(
        token in s
        for token in [
            "myeloid",
            "t_nk",
            "b_plasma",
            "endothelial",
            "oligodendrocyte",
            "mast",
            "erythroid",
        ]
    ):
        return "reference_or_nonmalignant_like"

    if any(
        token in s
        for token in [
            "mural",
            "pericyte",
            "fibroblast",
            "stromal",
            "ambiguous",
            "mixed",
        ]
    ):
        return "ambiguous_or_stromal"

    return "other"


if ANNOT_COL is not None:
    cmp["broad_biology"] = cmp[ANNOT_COL].map(broad_biology)
else:
    cmp["broad_biology"] = "annotation_unavailable"

display(
    cmp["broad_biology"]
    .value_counts(dropna=False)
    .rename_axis("broad_biology")
    .reset_index(name="n_cells")
)


In [ ]:
# =========================
# GROUP-LEVEL COMPARISON
# =========================
def compare_by_group(df, group_col):
    if group_col is None or group_col not in df.columns:
        return pd.DataFrame()

    tmp = df.copy()
    tmp[group_col] = tmp[group_col].astype(str)

    rows = []

    for group, g in tmp.groupby(group_col, observed=True):
        h_defined = g["historical_call"].isin(["aneuploid", "diploid"])
        c_defined = g["current_call"].isin(["aneuploid", "diploid"])

        h_aneu = (g["historical_call"] == "aneuploid")
        c_aneu = (g["current_call"] == "aneuploid")

        dip_to_aneu = (
            (g["historical_call"] == "diploid")
            & (g["current_call"] == "aneuploid")
        )

        aneu_to_dip = (
            (g["historical_call"] == "aneuploid")
            & (g["current_call"] == "diploid")
        )

        rows.append({
            group_col: group,
            "n_cells": len(g),
            "historical_defined": int(h_defined.sum()),
            "current_defined": int(c_defined.sum()),
            "historical_aneuploid_fraction_defined": (
                h_aneu.sum() / h_defined.sum()
                if h_defined.sum() else np.nan
            ),
            "current_aneuploid_fraction_defined": (
                c_aneu.sum() / c_defined.sum()
                if c_defined.sum() else np.nan
            ),
            "delta_aneuploid_fraction_defined_pp": (
                (
                    c_aneu.sum() / c_defined.sum()
                    if c_defined.sum() else np.nan
                )
                - (
                    h_aneu.sum() / h_defined.sum()
                    if h_defined.sum() else np.nan
                )
            ) * 100,
            "diploid_to_aneuploid": int(dip_to_aneu.sum()),
            "aneuploid_to_diploid": int(aneu_to_dip.sum()),
            "changed_cells": int(g["changed"].sum()),
            "changed_fraction": float(g["changed"].mean()),
        })

    out = pd.DataFrame(rows)

    return out.sort_values(
        ["delta_aneuploid_fraction_defined_pp", "n_cells"],
        ascending=[False, False],
    )


broad_compare = compare_by_group(cmp, "broad_biology")
cluster_compare = compare_by_group(cmp, CLUSTER_COL)
annotation_compare = compare_by_group(cmp, ANNOT_COL)
dataset_compare = compare_by_group(cmp, DATASET_COL)
condition_compare = compare_by_group(cmp, CONDITION_COL)

print("Broad biological groups")
display(broad_compare)

print("Clusters")
display(cluster_compare)

print("Annotations")
display(annotation_compare)

print("Datasets")
display(dataset_compare)

print("Conditions")
display(condition_compare)


In [ ]:
# =========================
# WHERE DO diploid -> aneuploid CHANGES OCCUR?
# =========================
d2a = cmp[
    (cmp["historical_call"] == "diploid")
    & (cmp["current_call"] == "aneuploid")
].copy()

print("diploid -> aneuploid cells:", len(d2a))

if CLUSTER_COL is not None:
    d2a_cluster = (
        d2a.groupby(CLUSTER_COL, observed=True)
        .size()
        .rename("n_diploid_to_aneuploid")
        .reset_index()
        .sort_values("n_diploid_to_aneuploid", ascending=False)
    )

    display(d2a_cluster)

if ANNOT_COL is not None:
    d2a_annotation = (
        d2a.groupby(ANNOT_COL, observed=True)
        .size()
        .rename("n_diploid_to_aneuploid")
        .reset_index()
        .sort_values("n_diploid_to_aneuploid", ascending=False)
    )

    display(d2a_annotation)

d2a_broad = (
    d2a.groupby("broad_biology", observed=True)
    .size()
    .rename("n_diploid_to_aneuploid")
    .reset_index()
    .sort_values("n_diploid_to_aneuploid", ascending=False)
)

display(d2a_broad)


In [ ]:
# =========================
# HISTORICAL VS CURRENT CLUSTER SUMMARY FILES
# =========================
cluster_key = (
    "leiden_scvi_main"
    if "leiden_scvi_main" in hist_cluster.columns
    and "leiden_scvi_main" in cur_cluster.columns
    else None
)

if cluster_key is not None:
    hcs = hist_cluster.copy()
    ccs = cur_cluster.copy()

    hcs[cluster_key] = hcs[cluster_key].astype(str)
    ccs[cluster_key] = ccs[cluster_key].astype(str)

    cluster_summary_file_compare = hcs.merge(
        ccs,
        on=cluster_key,
        how="outer",
        suffixes=("_historical", "_current"),
    )

    keep = [cluster_key]

    for c in [
        "n_cells",
        "n_aneuploid",
        "n_diploid",
        "n_not_defined",
        "n_missing",
        "aneuploid_fraction_defined",
    ]:
        for suffix in ["_historical", "_current"]:
            col = c + suffix
            if col in cluster_summary_file_compare.columns:
                keep.append(col)

    display(cluster_summary_file_compare[keep])
else:
    print("Could not directly merge cluster-summary files by leiden_scvi_main.")


In [ ]:
# =========================
# DECISION-ORIENTED READOUT
# =========================
def row_for(group_name):
    if len(broad_compare) == 0:
        return None

    x = broad_compare[
        broad_compare["broad_biology"] == group_name
    ]

    if len(x) == 0:
        return None

    return x.iloc[0]


tumor_row = row_for("tumor_like_or_cycling")
ref_row = row_for("reference_or_nonmalignant_like")
amb_row = row_for("ambiguous_or_stromal")

decision_rows = []

for label, row in [
    ("tumor_like_or_cycling", tumor_row),
    ("reference_or_nonmalignant_like", ref_row),
    ("ambiguous_or_stromal", amb_row),
]:
    if row is not None:
        decision_rows.append({
            "group": label,
            "n_cells": int(row["n_cells"]),
            "historical_aneuploid_fraction_defined": row[
                "historical_aneuploid_fraction_defined"
            ],
            "current_aneuploid_fraction_defined": row[
                "current_aneuploid_fraction_defined"
            ],
            "delta_pp": row[
                "delta_aneuploid_fraction_defined_pp"
            ],
            "diploid_to_aneuploid": int(
                row["diploid_to_aneuploid"]
            ),
            "changed_fraction": row[
                "changed_fraction"
            ],
        })

decision_table = pd.DataFrame(decision_rows)

print("Decision-oriented summary")
display(decision_table)

print("\nINTERPRETATION GUIDE")
print(
    "1) If the increase in aneuploid calls in the current run is concentrated "
    "mainly in tumor-like cells while reference/non-malignant-like compartments "
    "remain relatively stable, the current run may be biologically defensible."
)
print(
    "2) If aneuploid calls also increase broadly in myeloid, T/NK, endothelial, "
    "oligodendrocyte, or other reference-like populations, the current run may "
    "have lower specificity or a more permissive baseline."
)
print(
    "3) If tumor-like compartments are similar between historical and current "
    "runs but reference-like populations show substantially more aneuploid calls "
    "in the current run, the historical result is the more conservative source "
    "of malignant-CNV evidence."
)
print(
    "4) This comparison localizes and quantifies the disagreement. It does not, "
    "by itself, prove the mechanistic cause of the difference."
)


In [ ]:
# =========================
# FINAL READ-ONLY CHECKPOINT
# =========================
print("READ-ONLY AUDIT COMPLETE")
print("No project file was written or modified by this notebook.")
print()
print("Send me these outputs:")
print("  1) Overall transition matrix")
print("  2) Key transitions")
print("  3) Broad biological groups comparison")
print("  4) Top rows of cluster comparison")
print("  5) Top rows of annotation comparison")
print("  6) Decision-oriented summary")
